In [15]:
%pip install -U langchain-openai langgraph langchain
import os
from typing import TypedDict, Optional, Annotated, List, Dict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
# Imports for Real Wikipedia Search
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END


# Initialize the REAL Wikipedia Tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

ModuleNotFoundError: No module named 'langchain_community'

In [27]:
# 1. Setup Environment
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")
print(f"API_KEY: {my_api_key[:5]}...")

API_KEY: sk-pr...


In [ ]:

# Define the State
class AgentState(TypedDict):
    query: str
    search_results: Optional[str]
    summary: Optional[str]
    feedback: Optional[int]
    source: Optional[str]

# 2. Define Node Logic
def user_input_node(state: AgentState):
    print(f"\n--- Processing Query: {state['query']} ---")
    return state # Must return state to avoid NoneType errors

def book_search_agent(state: AgentState):
    print("--- Searching Local Library... ---")
    local_db = [
        "Deep Learning by Ian Goodfellow",
        "Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow",
        "Neural Networks and Deep Learning by Michael Nielsen"
    ]
    
    # Check if 'deep learning' is in the query
    if "deep learning" in state['query'].lower():
        result = local_db[0]
        source = "Local Library"
    else:
        result = "No local books found."
        source = "N/A"
        
    return {"search_results": result, "source": source}

def wikipedia_search_tool(state: AgentState):
    print("--- Routing to Wikipedia... ---")
    # 'wikipedia' simulation
    # result = "Wikipedia: Deep learning is part of a broader family of machine learning methods based on artificial neural networks."
    # return {"search_results": result, "source": "Wikipedia"}
    
    # This actually calls the Wikipedia API
    try:
        actual_result = wiki_tool.run(state['query'])
        return {"search_results": actual_result, "source": "Wikipedia"}
    except Exception as e:
        return {"search_results": f"Error fetching from Wiki: {str(e)}", "source": "Error"}
    

def summarize_agent(state: AgentState):
    print("---  LLM Summarizing... ---")
    api_key = os.getenv("OPENAI_API_KEY")
    
    # Initialize LLM with the key directly
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=my_api_key)
    
    prompt = f"Summarize this info for a student: {state['search_results']}"
    response = llm.invoke(prompt)
    
    return {"summary": response.content}

def output_node(state: AgentState):
    print(f"\n FINAL RESPONSE")
    print(f"Source: {state['source']}")
    print(f"Summary: {state['summary']}")
    return state

def feedback_node(state: AgentState):
    # Bonus: Simulate user rating
    rating = 5 
    print(f"\n---  Feedback Saved: {rating}/5 stars ---")
    return {"feedback": rating}

# --- Router Logic ---
def route_search_result(state: AgentState):
    """
    This function decides whether to go to the summarizer 
    or jump to Wikipedia.
    """
    if state["search_results"] == "No local books found.":
        return "wiki"
    return "summarize"

# 3. Build the Graph
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("user_input", user_input_node)
workflow.add_node("local_search", book_search_agent)
workflow.add_node("wiki_search", wikipedia_search_tool)
workflow.add_node("summarizer", summarize_agent)
workflow.add_node("output", output_node)
workflow.add_node("feedback", feedback_node)

# Define Connections
workflow.set_entry_point("user_input")
workflow.add_edge("user_input", "local_search")

# Conditional Routing: Local Search -> (Router) -> Wiki OR Summarizer
workflow.add_conditional_edges(
    "local_search",
    route_search_result,
    {
        "wiki": "wiki_search",
        "summarize": "summarizer"
    }
)

# Connect the rest
workflow.add_edge("wiki_search", "summarizer")
workflow.add_edge("summarizer", "output")
workflow.add_edge("output", "feedback")
workflow.add_edge("feedback", END)

# Compile
app = workflow.compile()

print(app.get_graph().draw_ascii())

             +-----------+        
             | __start__ |        
             +-----------+        
                    *             
                    *             
                    *             
             +------------+       
             | user_input |       
             +------------+       
                    *             
                    *             
                    *             
            +--------------+      
            | local_search |      
            +--------------+      
             ...         ...      
            .               .     
          ..                 ...  
+-------------+                 . 
| wiki_search |              ...  
+-------------+             .     
             ***         ...      
                *       .         
                 **   ..          
             +------------+       
             | summarizer |       
             +------------+       
                    *             
                    

In [36]:

# 4. Run the Graph
# Example 1: This will find a local book
print("RUNNING TEST 1...")
app.invoke({'query': 'Recommend a book about deep learning.'})

# Example 2: This will trigger the Wikipedia branch
print("\n" + "="*30 + "\n")
print("RUNNING TEST 2...")
app.invoke({'query': 'Tell me about Quantum Computing.'})

RUNNING TEST 1...

--- Processing Query: Recommend a book about deep learning. ---
--- Searching Local Library... ---
---  LLM Summarizing... ---

 FINAL RESPONSE
Source: Local Library
Summary: "Deep Learning" by Ian Goodfellow, Yoshua Bengio, and Aaron Courville is a comprehensive textbook that explores the field of deep learning, a subset of machine learning that focuses on neural networks with many layers. 

Key topics covered in the book include:

1. **Foundations**: The authors explain the fundamentals of machine learning and cover topics such as linear algebra, probability, and optimization that are essential for understanding deep learning.

2. **Neural Networks**: The book discusses the architecture and functioning of neural networks, including feedforward networks, convolutional networks, recurrent networks, and generative models.

3. **Training Deep Networks**: Techniques and challenges in training deep networks, such as overfitting, regularization, and optimization algorithm

{'query': 'Tell me about Quantum Computing.',
 'search_results': 'Wikipedia: Deep learning is part of a broader family of machine learning methods based on artificial neural networks.',
 'summary': 'Deep learning is a type of machine learning that uses artificial neural networks to analyze and process data. It is a more advanced approach within the broader field of machine learning.',
 'feedback': 5,
 'source': 'Wikipedia'}